# LAB 8

## Task 1: Teoría

### 1. Investigar el algoritmo AC-3 y su relación con el algoritmo de backtracking search

El algoritmo **AC-3 (Arc Consistency Algorithm #3)** es un método utilizado en problemas de satisfacción de restricciones (CSP) para reducir el espacio de búsqueda eliminando valores inconsistentes en las variables antes o durante la búsqueda. Su objetivo principal es lograr consistencia de arco, asegurando que para cada par de variables conectadas por una restricción, los valores asignados a una variable tengan al menos un valor compatible en la otra.

La relación con el algoritmo **Backtracking Search** radica en que AC-3 generalmente se utiliza como un paso previo o complemento a la búsqueda por backtracking. Aplicar AC-3 antes de iniciar la búsqueda ayuda a disminuir considerablemente la cantidad de decisiones incorrectas que el algoritmo de backtracking podría tomar, reduciendo el tamaño del árbol de búsqueda y mejorando significativamente la eficiencia del proceso.

### 2. Defina en sus propias palabras el término “Arc Consistency”

El término **Arc Consistency (Consistencia de Arco)** en un problema CSP se refiere al estado en el cual todas las variables conectadas mediante restricciones han eliminado los valores incompatibles. En otras palabras, una variable es arc-consistente respecto a otra cuando cada uno de sus posibles valores tiene al menos una opción compatible en la variable vecina. Lograr consistencia de arco implica garantizar que ninguna variable tenga valores imposibles o incompatibles, facilitando así encontrar soluciones válidas para el problema.


In [4]:
# %% [code]
import time
import random

# Definición de exámenes y días disponibles
exams = ["Exam1", "Exam2", "Exam3", "Exam4", "Exam5", "Exam6", "Exam7"]
days = ["Lunes", "Martes", "Miércoles"]

# Inscripción de los estudiantes: cada uno con 3 exámenes.
students = {
    "A": ["Exam1", "Exam2", "Exam3"],
    "B": ["Exam3", "Exam4", "Exam5"],
    "C": ["Exam5", "Exam6", "Exam7"],
    "D": ["Exam1", "Exam4", "Exam6"]
}

# Construir el grafo de conflictos: dos exámenes entran en conflicto si algún alumno está inscrito en ambos.
# Esto garantiza que para cada alumno no se asignen dos exámenes el mismo día.
conflicts = {exam: set() for exam in exams}
for student, exam_list in students.items():
    for i in range(len(exam_list)):
        for j in range(i + 1, len(exam_list)):
            exam_i = exam_list[i]
            exam_j = exam_list[j]
            conflicts[exam_i].add(exam_j)
            conflicts[exam_j].add(exam_i)

# Función para verificar si asignar 'value' a 'var' es consistente con la asignación parcial.
def is_consistent(assignment, var, value):
    for neighbor in conflicts[var]:
        if neighbor in assignment and assignment[neighbor] == value:
            return False
    return True

# Función para imprimir el horario de exámenes por alumno.
# Además, verifica que para cada alumno los días asignados sean distintos (cumpliendo la restricción).
def print_student_schedule(solution, algo_name):
    print(f"{algo_name} - Horario de exámenes por estudiante:")
    if solution is None:
        print("  No se encontró solución.")
    else:
        for student, exam_list in students.items():
            print(f"Estudiante {student}:")
            assigned_days = []
            for exam in exam_list:
                day_assigned = solution.get(exam, "Sin asignación")
                assigned_days.append(day_assigned)
                print(f"  {exam}: {day_assigned}")
            if len(assigned_days) != len(set(assigned_days)):
                print("  --> ¡Conflicto! El estudiante tiene más de un examen el mismo día.")
            else:
                print("  --> Horario sin conflictos.")
    print("\n" + "="*40 + "\n")

# ------------------------------------------------------------
# 1. Algoritmo de Backtracking
def backtracking(assignment):
    if len(assignment) == len(exams):
        return assignment
    unassigned = [e for e in exams if e not in assignment]
    var = unassigned[0]
    for value in days:
        if is_consistent(assignment, var, value):
            assignment[var] = value
            result = backtracking(assignment)
            if result is not None:
                return result
            del assignment[var]  # Retroceso
    return None

start_bt = time.time()
bt_solution = backtracking({})
bt_time = time.time() - start_bt
print("Backtracking solution:", bt_solution)
print("Tiempo (segundos):", bt_time, "\n")

# ------------------------------------------------------------
# 2. Algoritmo de Beam Search
def beam_search(beam_width):
    initial_state = {}
    beam = [initial_state]
    while beam:
        new_beam = []
        for state in beam:
            if len(state) == len(exams):
                return state
            unassigned = [e for e in exams if e not in state]
            var = unassigned[0]
            for value in days:
                if is_consistent(state, var, value):
                    new_state = state.copy()
                    new_state[var] = value
                    new_beam.append(new_state)
        if not new_beam:
            return None
        new_beam.sort(key=lambda s: len(s), reverse=True)
        beam = new_beam[:beam_width]
    return None

start_beam = time.time()
beam_solution = beam_search(beam_width=3)
beam_time = time.time() - start_beam
print("Beam Search solution:", beam_solution)
print("Tiempo (segundos):", beam_time, "\n")

# ------------------------------------------------------------
# 3. Algoritmo de Local Search (Min-Conflicts)
def count_conflicts(assignment, var, value):
    count = 0
    for neighbor in conflicts[var]:
        if neighbor in assignment and assignment[neighbor] == value:
            count += 1
    return count

def min_conflicts(max_iter=1000):
    assignment = {exam: random.choice(days) for exam in exams}
    for _ in range(max_iter):
        conflicted = []
        for exam in exams:
            if any(assignment[exam] == assignment[neighbor] for neighbor in conflicts[exam] if neighbor in assignment):
                conflicted.append(exam)
        if not conflicted:
            return assignment
        var = random.choice(conflicted)
        best_value = None
        best_conflict = float('inf')
        for value in days:
            c = count_conflicts(assignment, var, value)
            if c < best_conflict:
                best_conflict = c
                best_value = value
        assignment[var] = best_value
    return None

start_local = time.time()
local_solution = min_conflicts(max_iter=1000)
local_time = time.time() - start_local
print("Local Search solution:", local_solution)
print("Tiempo (segundos):", local_time, "\n")

# ------------------------------------------------------------
# Imprimir el horario de exámenes por estudiante para cada algoritmo.
print_student_schedule(bt_solution, "Backtracking")
print_student_schedule(beam_solution, "Beam Search")
print_student_schedule(local_solution, "Local Search")


Backtracking solution: {'Exam1': 'Lunes', 'Exam2': 'Martes', 'Exam3': 'Miércoles', 'Exam4': 'Martes', 'Exam5': 'Lunes', 'Exam6': 'Miércoles', 'Exam7': 'Martes'}
Tiempo (segundos): 7.700920104980469e-05 

Beam Search solution: {'Exam1': 'Lunes', 'Exam2': 'Martes', 'Exam3': 'Miércoles', 'Exam4': 'Martes', 'Exam5': 'Lunes', 'Exam6': 'Miércoles', 'Exam7': 'Martes'}
Tiempo (segundos): 0.000408172607421875 

Local Search solution: {'Exam1': 'Martes', 'Exam2': 'Miércoles', 'Exam3': 'Lunes', 'Exam4': 'Miércoles', 'Exam5': 'Martes', 'Exam6': 'Lunes', 'Exam7': 'Miércoles'}
Tiempo (segundos): 0.00015497207641601562 

Backtracking - Horario de exámenes por estudiante:
Estudiante A:
  Exam1: Lunes
  Exam2: Martes
  Exam3: Miércoles
  --> Horario sin conflictos.
Estudiante B:
  Exam3: Miércoles
  Exam4: Martes
  Exam5: Lunes
  --> Horario sin conflictos.
Estudiante C:
  Exam5: Lunes
  Exam6: Miércoles
  Exam7: Martes
  --> Horario sin conflictos.
Estudiante D:
  Exam1: Lunes
  Exam4: Martes
  Exam6: